# 08 · Evaluation report — gates G3 & G4, success criteria (PRD §13, §20.1, §22)
**Runs:** locally. **Needs:** notebooks 05–07. **Produces:** `reports/evaluation_report.json`.

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

In [ ]:
import json, time
import joblib
import pandas as pd
from app.ml.calibration.conformal import ConformalPredictor
from app.ml.agents.fusion import FusionModel
from vera_training import calibration, evaluation
from vera_training.agreement import get_reference_agreement

cache = joblib.load(paths.FUSION_CACHE)["matrices"]
comparison = json.loads((paths.REPORTS / "cv_comparison.json").read_text())
fusion = FusionModel(paths.MODELS / "fusion_ensemble.json")
conformal = json.loads((paths.MODELS / "conformal_qhat.json").read_text())

members = fusion.predict_members(cache["test"]["X"])
coverage = calibration.interval_metrics(members.mean(0).clip(1, 5), members.std(0),
                                        cache["test"]["y"], conformal)
g4 = evaluation.gate_g4(comparison, coverage["coverage"])
print(json.dumps(g4, indent=2))

## Per-aspect detection F1 / sentiment accuracy on the test split (target F1 ≥ 0.70)

In [ ]:
test_frame = cache["test"]["frame"]
aspects = evaluation.aspect_metrics(test_frame.aspects.tolist(), cache["test"]["aspects"])
aspects

## G3 — full production pipeline on 10 reviews (target < 15 s each on CPU)

In [ ]:
from app.ml.pipeline import VERAPipeline

pipeline = VERAPipeline(paths.MODELS)
latencies = []
for row in test_frame.head(10).itertuples():
    started = time.perf_counter()
    pipeline.process(text=row.text, star_rating=float(row.rating))
    latencies.append(time.perf_counter() - started)
g3 = {"gate": "G3", "passed": max(latencies) < 15, "max_seconds": max(latencies),
      "mean_seconds": sum(latencies) / len(latencies)}
print(g3)

In [ ]:
report = {
    "cv_comparison": comparison,
    "test_interval": coverage,
    "conformal": conformal,
    "aspects": aspects.to_dict(orient="records"),
    "gates": {"G3": g3, "G4": g4},
    "aspect_f1_all_at_least_0_70": bool((aspects.detection_f1 >= 0.70).all()),
    "label_noise_reference": get_reference_agreement(
        json.loads(paths.AGREEMENT_REPORT.read_text())
    ),
}
(paths.REPORTS / "evaluation_report.json").write_text(json.dumps(report, indent=2, default=str))
print("Wrote", paths.REPORTS / "evaluation_report.json")